# Salis Lab Promoter Calculator — Example

![Salis Lab Promoter Calculator — Example](https://proto-bio.github.io/proto-assets/images/tool/promoter_calculator/hero.png)

This notebook demonstrates `run_promoter_calculator`, which predicts sigma70 promoter strength on DNA sequences in *E. coli*. The calculator scans both strands for canonical sigma70 elements (UP, -35 hexamer, spacer, -10 hexamer, discriminator) and returns the binding free energy `dG_total` (kcal/mol) and predicted transcription initiation rate `Tx_rate` (au) for each candidate promoter.

Reference: LaFleur et al., *Nature Communications* 2022, [DOI 10.1038/s41467-022-32829-5](https://doi.org/10.1038/s41467-022-32829-5).

In [1]:
from proto_tools.tools.gene_annotation.promoter_calculator import (
    PromoterCalculatorConfig,
    PromoterCalculatorInput,
    run_promoter_calculator,
)
from proto_tools.utils.notebook_docs import display_api_reference

## Input API Reference

In [2]:
display_api_reference("promoter_calculator", "input")

**Input** — `PromoterCalculatorInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | DNA sequences to scan for E. coli sigma70 (housekeeping) promoters |

## Config API Reference

In [3]:
display_api_reference("promoter_calculator", "config")

**Config** — `PromoterCalculatorConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>threads</code> | <code>int</code> | <code>1</code> | CPU threads for promoter calculator parallelism; raise on multi-core hosts |
| <code>circular</code> | <code>bool</code> | <code>False</code> | Examine the wraparound junction (use for plasmids/circular genomes) |

## Output API Reference

`PromoterCalculatorOutput.results` is a `list[PromoterCalculatorSequenceResult]`, one per input sequence. Each `PromoterCalculatorSequenceResult` carries:

- `sequence_id: str` — the input sequence identifier
- `predictions: list[PromoterPrediction]` — every predicted promoter on both strands

Each `PromoterPrediction` describes a single predicted promoter, including its transcription start site, strand, binding free energy `dG_total`, predicted transcription initiation rate `Tx_rate`, promoter sequence, and the bounds of the UP, -35 hexamer, spacer, -10 hexamer, and discriminator elements.

In [4]:
display_api_reference("promoter_calculator", "output")

**Output** — `PromoterCalculatorOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[PromoterCalculatorSequenceResult]</code> | <code>[]</code> | Per-sequence promoter predictions |

**`PromoterCalculatorSequenceResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence_id</code> | <code>str</code> | required | ID of the input sequence |
| <code>predictions</code> | <code>list[PromoterPrediction]</code> | <code>[]</code> | Predicted promoters across both strands |

**`PromoterPrediction`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>tss_name</code> | <code>str</code> | required | Promoter label, e.g. 'Fwd123' or 'Rev456' |
| <code>tss</code> | <code>int</code> | required | 0-indexed transcription start site on the input sequence |
| <code>strand</code> | <code>Literal['+', '-']</code> | required | '+' = forward strand, '-' = reverse-complement |
| <code>dG_total</code> | <code>float</code> | required | Predicted binding free energy in kcal/mol |
| <code>Tx_rate</code> | <code>float</code> | required | Predicted transcription initiation rate in arbitrary units (model-relative) |
| <code>promoter_sequence</code> | <code>str</code> | required | DNA spanning the predicted promoter |
| <code>length</code> | <code>int</code> | required | Length of the promoter sequence in nt |
| <code>UP_position</code> | <code>list[int]</code> | required | UP element 0-indexed [start, end_inclusive] on the input sequence |
| <code>hex35_position</code> | <code>list[int]</code> | required | -35 hexamer 0-indexed [start, end_inclusive] |
| <code>spacer_position</code> | <code>list[int]</code> | required | Spacer 0-indexed [start, end_inclusive] |
| <code>hex10_position</code> | <code>list[int]</code> | required | -10 hexamer 0-indexed [start, end_inclusive] |
| <code>disc_position</code> | <code>list[int]</code> | required | Discriminator 0-indexed [start, end_inclusive] |

In [5]:
# lacUV5 promoter (well-characterized E. coli sigma70 promoter)
lac_uv5 = "AAAATTGTGAGCGGATAACAATTTCACACAGGAAACAGCTATGACC"
sequence = "A" * 20 + lac_uv5 + "A" * 20

inputs = PromoterCalculatorInput(sequences=[sequence])
config = PromoterCalculatorConfig(threads=1)
result = run_promoter_calculator(inputs, config)

print(f"Predicted promoters: {result.results[0].num_promoters}")
for pred in result.results[0].predictions[:3]:
    print(f"  {pred.tss_name}  strand={pred.strand}  "
          f"dG={pred.dG_total:.2f} kcal/mol  Tx_rate={pred.Tx_rate:.0f}")

Running run_promoter_calculator [00:00]

Predicted promoters: 5
  Rev61  strand=-  dG=-1.83 kcal/mol  Tx_rate=838
  Rev66  strand=-  dG=-1.63 kcal/mol  Tx_rate=607
  Fwd66  strand=+  dG=-1.48 kcal/mol  Tx_rate=475


In [6]:
# Compare two promoter variants in batch, with circular DNA handling
variants = {
    "J23119_strong": "TTGACAGCTAGCTCAGTCCTAGGTATAATGCTAGC",
    "J23113_weak":   "CTGATGGCTAGCTCAGTCCTAGGGATTATGCTAGC",
}
inputs = PromoterCalculatorInput(
    sequences=["A" * 30 + s + "A" * 30 for s in variants.values()],
)
config = PromoterCalculatorConfig(threads=4, circular=False)
result = run_promoter_calculator(inputs, config)

for seq_result in result.results:
    best = max(seq_result.predictions, key=lambda p: p.Tx_rate, default=None)
    if best is None:
        print(f"{seq_result.sequence_id}: no promoter detected")
    else:
        print(f"{seq_result.sequence_id}: best Tx_rate={best.Tx_rate:.0f} "
              f"(dG={best.dG_total:.2f}, strand={best.strand})")

Running run_promoter_calculator [00:00]

seq_0: best Tx_rate=83433 (dG=-4.64, strand=+)
seq_1: best Tx_rate=3522 (dG=-2.71, strand=+)


In [7]:
from pathlib import Path

out_dir = Path("./promoter_results")
out_dir.mkdir(exist_ok=True)
result.export(name="promoters", export_path=str(out_dir), file_format="csv")
result.export(name="promoters", export_path=str(out_dir), file_format="json")

print("Wrote:", sorted(p.name for p in out_dir.iterdir()))

Wrote: ['promoters.csv', 'promoters.json']
